# VoxPrep HK — Kaggle QLoRA training and export

**Unexecuted notebook: no training or human review is claimed.** Use Kaggle first; free Colab is a fallback and GPU availability is not guaranteed.

Before running: complete Kaggle **phone verification**, enable **Internet**, and select a CUDA GPU in Session options. A T4 is sufficient for the intended QLoRA setup; the script uses GPU 0 only even if two T4s appear. T4 does **not** support BF16: `ml/train.py` detects this and uses FP16. Qwen3-4B weights require roughly **8 GB** of downloads, plus CUDA packages/cache, adapters/checkpoints and conversion space. Check free disk space and the account's displayed GPU quota. No paid compute is requested.

Commit the real, merged `ml/data/reviewed_examples.jsonl` to the public fork branch selected below **after** WS1/WS2/WS3/WS5 are integrated. Only that source file is needed; this notebook makes seed-42 splits. Never fabricate reviews or turn off data gates. Do not put private applicant/employer information or tokens in Git.

References: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [phone/GPU access](https://www.kaggle.com/product-feedback/268333), [free Colab availability](https://research.google.com/colaboratory/faq.html).

## 1. Clone a specific public fork branch

Set `BRANCH` to the published branch containing all implementation changes and the final human-reviewed corpus. The default is the fork's `main`, **not** the upstream repository. An existing checkout must match the requested URL/branch and be clean; this cell does not overwrite it. The exact source commit is recorded for reproducibility. No manual source upload or Kaggle API token is needed.

This cell prefers, in order: a private dataset containing this repository (attach it with `scripts/kaggle_train.py push --dataset`), a private dataset containing the human-reviewed `reviewed_examples.jsonl` (upload it with `scripts/kaggle_train.py dataset`), and only then the public fork branch cloned below. Attached inputs are copied into the working directory so `python -m ml.dataset` runs unchanged, and the corpus is checked against the SHA-256 recorded in the attached `dataset_manifest.json`. When the repository comes from a dataset there is no `.git` history, so the dataset name is recorded instead of a commit hash.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = "https://github.com/kelvin0611/COMP4431_GroupProject_InterviewCoach.git"
BRANCH = "main"  # Change if reviewed training data is published on another branch.
ATTACHED_DATA_SLUG = "voxprep-hk-reviewed"  # Private dataset from scripts/kaggle_train.py
ATTACHED_REPO_SLUG = None  # Set when this repository itself is attached as a dataset
INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else None
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
PROJECT = WORK / "COMP4431_GroupProject_InterviewCoach"
WORK.mkdir(parents=True, exist_ok=True)


def attached(slug):
    """Attached dataset directory for a slug, matched case-insensitively."""
    if INPUT is None or not slug or not INPUT.is_dir():
        return None
    wanted = slug.rsplit("/", 1)[-1].lower()
    for path in sorted(INPUT.iterdir()):
        if path.is_dir() and path.name.lower() == wanted:
            return path
    return None


def attached_with(*names):
    """Attached dataset directories that contain every one of these files."""
    if INPUT is None:
        return []
    return sorted(
        path
        for path in INPUT.iterdir()
        if path.is_dir() and all((path / name).is_file() for name in names)
    )


def same_code(source):
    """True when PROJECT already holds exactly the attached code (keeps a resume)."""
    names = ("ml/train.py", "ml/config.json", "ml/dataset.py")
    return all(
        (PROJECT / name).is_file()
        and (PROJECT / name).read_bytes() == (source / name).read_bytes()
        for name in names
    )


# 1. Code: an attached copy of this repository wins; otherwise clone the branch.
repo_source = attached(ATTACHED_REPO_SLUG)
if repo_source is None:
    candidates = attached_with("ml/train.py", "ml/config.json")
    assert len(candidates) <= 1, f"Attach at most one copy of this repo: {candidates}"
    repo_source = candidates[0] if candidates else None
REPO_LABEL = REPO
SOURCE_COMMIT = None
if repo_source is not None:
    if not PROJECT.exists() or not same_code(repo_source):
        if PROJECT.exists():
            shutil.rmtree(PROJECT)
        shutil.copytree(repo_source, PROJECT)
    REPO_LABEL = f"kaggle-input/{repo_source.name}"
    if (PROJECT / ".git").is_dir():
        SOURCE_COMMIT = subprocess.check_output(
            ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
        ).strip()
    print("Training source:", repo_source, "commit:", SOURCE_COMMIT)
    # Record the attached dataset as the source instead of a branch that was not used.
    REPO, BRANCH = REPO_LABEL, None
else:
    if not PROJECT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--single-branch",
                "--branch",
                BRANCH,
                REPO,
                str(PROJECT),
            ],
            check=True,
        )
    else:
        assert (
            subprocess.check_output(
                ["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True
            ).strip()
            == REPO
        )
        assert (
            subprocess.check_output(
                ["git", "-C", str(PROJECT), "branch", "--show-current"], text=True
            ).strip()
            == BRANCH
        )
        assert not subprocess.check_output(
            ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
        ).strip(), "Existing checkout has edits; inspect them before continuing."
    SOURCE_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip()
    print("Training source:", REPO, BRANCH, SOURCE_COMMIT)
os.chdir(PROJECT)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 2. Data: a private attached dataset holding the human-reviewed corpus.
DATA_FILE = "reviewed_examples.jsonl"


def input_listing():
    """Names under /kaggle/input, printed so a missing mount is provable in the log."""
    if INPUT is None or not INPUT.is_dir():
        return "no /kaggle/input"
    return sorted(path.name for path in INPUT.iterdir())


def corpus_dir():
    """Directory holding reviewed_examples.jsonl, found by content, not by name.

    An attached dataset normally mounts as /kaggle/input/<slug>/, but a versioned or
    renamed mount must still be usable, so search the whole input tree for the file.
    """
    if INPUT is None or not INPUT.is_dir():
        return None
    found = sorted(INPUT.rglob(DATA_FILE))
    assert len(found) <= 1, f"Attach exactly one reviewed corpus; found {found}"
    return found[0].parent if found else None


print("Attached inputs:", input_listing())
data_source = attached(ATTACHED_DATA_SLUG)
if data_source is not None and not (data_source / DATA_FILE).is_file():
    data_source = None
if data_source is None:
    data_source = corpus_dir()
# Kaggle can take a minute to mount a dataset attached to a fresh kernel version.
for attempt in range(1, 31):
    if data_source is not None:
        break
    print(f"Waiting for the attached corpus ({attempt}/30):", input_listing(), flush=True)
    time.sleep(10)
    data_source = attached(ATTACHED_DATA_SLUG) or corpus_dir()
print("Data source:", data_source, flush=True)
if data_source is not None:
    source = data_source / DATA_FILE
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    manifest_file = data_source / "dataset_manifest.json"
    if manifest_file.is_file():
        recorded = json.loads(manifest_file.read_text())["files"][source.name]["sha256"]
        assert digest == recorded, (
            "Attached corpus does not match its own dataset_manifest.json."
        )
        print("Reviewed corpus sha256:", digest, "(matches dataset_manifest.json)")
    else:
        print("Reviewed corpus sha256:", digest)
    target = PROJECT / "ml/data/reviewed_examples.jsonl"
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.resolve() != source.resolve():
        shutil.copy2(source, target)
    DATA_SOURCE = f"kaggle-input/{data_source.name}"
else:
    DATA_SOURCE = REPO_LABEL if BRANCH is None else f"{REPO_LABEL} ({BRANCH})"

reviewed = PROJECT / "ml/data/reviewed_examples.jsonl"
assert reviewed.is_file(), (
    "No reviewed corpus. Attach the PRIVATE dataset holding reviewed_examples.jsonl "
    "(scripts/kaggle_train.py dataset); /kaggle/input held "
    f"{input_listing()}. Do not use drafts or create approvals here."
)
with reviewed.open(encoding="utf-8") as handle:
    rows = sum(1 for line in handle if line.strip())
print("Reviewed corpus:", DATA_SOURCE, "rows:", rows)

## 2. Install the recorded Python/CUDA dependencies

Install the repository's software dependencies and pinned training stack. Commands use `check=True`: a failed installation or data validation stops the cell. If the notebook asks for a kernel restart, restart and rerun the clone cell before continuing. GPU training has not been executed during development of this notebook.

The stock Kaggle image ships transformers 5.x, which breaks bitsandbytes at import; `ml/requirements-train.txt` therefore pins transformers/peft/accelerate/bitsandbytes, and this cell asserts the imported versions afterwards. It keeps the image's own driver-matched torch unless `INSTALL_TORCH` is enabled, because changing torch needs its own verified run.

In [ ]:
INSTALL_TORCH = False  # Keep Kaggle's driver-matched torch until its own test passes.
# The backend lock (requirements.lock) is the API stack, not the training stack: installing
# it would pull FastAPI/psycopg into the GPU image. Training deps are the pins below.
pins = [
    line.strip()
    for line in Path("ml/requirements-train.txt").read_text().splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
packages = [pin for pin in pins if INSTALL_TORCH or not pin.startswith("torch==")]
# The stock image ships transformers 5.x, which breaks bitsandbytes; these pins fix it.
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)

import importlib.metadata

import torch

expected = {
    name: version for name, version in (pin.split("==", 1) for pin in pins if "==" in pin)
}
for name, version in expected.items():
    if name == "torch" and not INSTALL_TORCH:
        print(f"Kept the image torch {torch.__version__} (pinned {version}).")
        continue
    installed = importlib.metadata.version(name)
    assert installed == version, (
        f"{name} is {installed}; ml/requirements-train.txt requires {version}"
    )
print("Training stack:", {name: importlib.metadata.version(name) for name in expected})

## 3. Validate review evidence and create seed-42 splits

Every row requires current human approval; the assigned stratified sample requires two independent reviewers. Duplicates and cross-split source/template/role groups are rejected. The existing splitter records seed 42. Confirm the real corpus count matches `expected_examples` (planned WS2 count: **480**). For an intentionally smaller reviewed corpus, change the committed config beforehand and report the actual size; do not silently resize it here. The 2,048-token and assistant-only masking gates remain in `train.py`.

In [ ]:
import json

config = json.loads(Path("ml/config.json").read_text())
assert config["seed"] == 42, "This notebook's split protocol uses seed 42."
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.dataset",
        "validate",
        "--input",
        "ml/data/reviewed_examples.jsonl",
        "--require-review",
        "--check-duplicates",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.dataset",
        "split",
        "--input",
        "ml/data/reviewed_examples.jsonl",
        "--output",
        config["data_dir"],
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "ml.train", "--config", "ml/config.json", "--validate-only"],
    check=True,
)

## 4. Confirm CUDA and train

The configuration uses NF4 double quantization, rank 32/alpha 64, batch 1 × accumulation 16, evaluation/checkpointing every 10 optimizer steps, logging every 2, and best validation checkpoint selection. T4 uses FP16 automatically; BF16-capable GPUs use BF16. There is no CPU fallback. Set `RESUME` only to a retained checkpoint from this same run/config if recovering from a disconnection. Keep all failures and actual logs; a loss curve alone does not establish model quality.

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU in Kaggle/Colab session options."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Precision:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")
RESUME = None  # Example: ml/runs/voxprep-qlora/checkpoints/checkpoint-20
command = [sys.executable, "-m", "ml.train", "--config", "ml/config.json"]
if RESUME:
    command += ["--resume-from-checkpoint", RESUME]
subprocess.run(command, check=True)
run = Path(config["output_dir"])
(run / "source_revision.json").write_text(
    json.dumps({"repo": REPO, "branch": BRANCH, "commit": SOURCE_COMMIT}, indent=2)
)

## 5. Inspect genuine evidence

The completed manifest, adapter hashes, loss plot and log must actually exist. Inspect the curves rather than inserting example results. The manifest is the source of training status; without a completed manifest the app stays `not_trained`.

In [ ]:
import hashlib

from IPython.display import Image, display

manifest = json.loads((run / "model_manifest.json").read_text())
assert manifest["status"] == "training_completed"
for name, digest in manifest["adapter_sha256"].items():
    assert hashlib.sha256((run / "adapter" / name).read_bytes()).hexdigest() == digest
assert (run / "training_log.jsonl").stat().st_size > 0
display(Image(filename=str(run / "loss.png")))
print("Best validation loss:", manifest["best_validation_loss"])

## 6. Export for the app — default route B

**Diagnostic route-A preflight:** official llama.cpp supports `convert_lora_to_gguf.py … --base … --outfile … --outtype f16` and `llama-server -m base.gguf --lora adapter.gguf`. Converter `--base` needs the matching pinned HF configuration, not base weights. Hot LoRA and quantized inference are supported upstream, but this project's actual Qwen adapter + Q4_K_M parity is **unverified**: development has no adapter, llama-server or converter dependencies. We could not start an end-to-end test within the plan's 30-minute cap; route B is selected. Do not label A verified merely because the flags exist.

`EXPORT_MODE = "B"` reloads the pinned base in FP16, merges the verified adapter, converts to F16 GGUF, then quantizes to Q4_K_M **on Kaggle**. Only download the final GGUF (roughly 2.5 GB; inspect actual size) plus the run archive. The disposable intermediate merged/F16 files are removed only after successful conversion/checksums to fit notebook output limits. For A after a real smoke test, choose `"A"`; it saves only the small GGUF adapter. Use `"skip"` to package evidence without conversion.

The official llama.cpp checkout revision is recorded. To repeat an export, set `LLAMA_CPP_REVISION` to that recorded commit. Build CPU-only conversion tools so it also works on T4 without CUDA compiler setup. Export is unexecuted here and can take substantial time/disk space.

References: [converter](https://github.com/ggml-org/llama.cpp/blob/master/convert_lora_to_gguf.py), [server flags](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md), [LoRA inference test](https://github.com/ggml-org/llama.cpp/blob/master/tests/test-lora-conversion-inference.sh).

In [ ]:
EXPORT_MODE = (
    "B"  # "A" only after real parity testing; "skip" keeps training evidence only.
)
LLAMA_CPP_REVISION = "master"  # Replace with recorded commit for repeat exports.
assert EXPORT_MODE in {"A", "B", "skip"}
if EXPORT_MODE != "skip":
    llama = WORK / "llama.cpp"
    if not llama.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ggml-org/llama.cpp.git",
                str(llama),
            ],
            check=True,
        )
    subprocess.run(
        [
            "git",
            "-C",
            str(llama),
            "fetch",
            "--depth",
            "1",
            "origin",
            LLAMA_CPP_REVISION,
        ],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(llama), "checkout", "--detach", "FETCH_HEAD"], check=True
    )
    llama_revision = subprocess.check_output(
        ["git", "-C", str(llama), "rev-parse", "HEAD"], text=True
    ).strip()
    (run / "llama_cpp_revision.txt").write_text(llama_revision + "\n")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(llama / "requirements/requirements-convert_hf_to_gguf.txt"),
        ],
        check=True,
    )
    if EXPORT_MODE == "A":
        from huggingface_hub import snapshot_download

        pinned_config = snapshot_download(
            manifest["base_model"],
            revision=manifest["base_revision"],
            allow_patterns=["config.json", "tokenizer*", "special_tokens_map.json"],
        )
        adapter_gguf = run / "adapter.gguf"
        subprocess.run(
            [
                sys.executable,
                str(llama / "convert_lora_to_gguf.py"),
                str(run / "adapter"),
                "--base",
                pinned_config,
                "--outfile",
                str(adapter_gguf),
                "--outtype",
                "f16",
            ],
            check=True,
        )
        (run / "adapter_gguf_sha256.txt").write_text(
            hashlib.sha256(adapter_gguf.read_bytes()).hexdigest() + "\n"
        )
        print(
            "Route A conversion complete; real base+adapter inference validation still required."
        )
    else:
        subprocess.run(
            [
                "cmake",
                "-S",
                str(llama),
                "-B",
                str(llama / "build"),
                "-DGGML_CUDA=OFF",
                "-DLLAMA_CURL=OFF",
            ],
            check=True,
        )
        subprocess.run(
            [
                "cmake",
                "--build",
                str(llama / "build"),
                "--target",
                "llama-quantize",
                "-j",
                "2",
            ],
            check=True,
        )
        export_dir = WORK / "voxprep-export"
        subprocess.run(
            [
                sys.executable,
                "-m",
                "ml.export",
                "--run",
                str(run),
                "--output",
                str(export_dir),
                "--llama-cpp",
                str(llama),
                "--quantize-bin",
                str(llama / "build/bin/llama-quantize"),
            ],
            check=True,
        )
        exported = json.loads((export_dir / "export_manifest.json").read_text())
        final_gguf = export_dir / "voxprep-Q4_K_M.gguf"
        digest = hashlib.sha256()
        with final_gguf.open("rb") as file:
            for block in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(block)
        assert digest.hexdigest() == exported["files"][final_gguf.name]
        import shutil

        shutil.copy2(export_dir / "export_manifest.json", run / "export_manifest.json")
        # Only disposable artifacts created by this export, after final checksum validation.
        shutil.rmtree(export_dir / "merged")
        (export_dir / "voxprep-f16.gguf").unlink()
        print(
            "Download final GGUF separately:",
            final_gguf,
            "GiB:",
            round(final_gguf.stat().st_size / 2**30, 2),
        )

## 7. Download evidence and connect the app

The last code cell archives the entire run: `adapter/`, `loss.png`, `training_log.jsonl`, `model_manifest.json`, split/config/environment records, checkpoints and export/source revision evidence. Kaggle exposes the ZIP and (route B) GGUF under **Output**; save a notebook version with outputs and download both. Colab triggers a ZIP download and provides a GGUF link.

Extract the archive **into** `ml/runs/voxprep-qlora/` locally. Keep the base model separately for evaluation.

- **B:** set `MODEL_PROVIDER=llama` and `MODEL_PATH=models/voxprep-Q4_K_M.gguf`; leave `LORA_PATH` unset. Run `make dev` or `make demo` after stopping your previous model service explicitly.
- **A (after smoke validation):** set `MODEL_PATH` to the matching base Q4_K_M and `LORA_PATH=ml/runs/voxprep-qlora/adapter.gguf`. The launcher checks the adapter exists and refuses to reuse a running service without it active.
- Run `python -m evaluation.metadata` to refresh the saved training fields. The Methodology endpoint also reads the actual local manifest; missing/incomplete manifests keep `not_trained`. `adapter_available` separately indicates local artifact presence. Training completion is not an evaluation pass.
- Follow `docs/EVALUATION.md` / WS4 for the **entire real held-out test split**, with actual sample counts. Preserve base vs fine-tuned failures and human ratings; do not pad to 120 prompts or use training examples.

In [ ]:
import shutil

from IPython.display import FileLink, display

required = [
    run / "adapter/adapter_model.safetensors",
    run / "loss.png",
    run / "training_log.jsonl",
    run / "model_manifest.json",
]
assert all(path.is_file() for path in required), (
    "Incomplete run: do not package as completed."
)
archive = shutil.make_archive(
    str(WORK / "voxprep-training-evidence"), "zip", root_dir=run
)
print("Download and retain:", archive)
display(FileLink(os.path.relpath(archive, WORK)))
if EXPORT_MODE == "B":
    display(FileLink(os.path.relpath(final_gguf, WORK)))
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Kaggle: save the notebook version and download ZIP/GGUF from Output.")